# Stage 3 — Colon Cancer Classifier (EfficientNet-B0)

**Task:** Binary classification — `colon_aca` (cancer) vs `colon_n` (normal)
**Dataset:** LC25000 colon subset
**Model:** EfficientNet-B0 pretrained on ImageNet

## Cell 1 — Check GPU & Install Dependencies

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
!pip install -q datasets timm scikit-learn matplotlib seaborn grad-cam

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/colorectal_cad/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Saving to: {SAVE_DIR}')

## Cell 3 — Clone Repository

In [ ]:
!git clone https://github.com/ashwajeetgedam12/histopath-crc-detection
%cd colorectal-cancer-cad-pipeline
!git log --oneline -5

## Cell 4 — Configuration

In [ ]:
import torch

CONFIG = {
    'dataset_name'  : '1aurent/LC25000',
    'train_ratio'   : 0.70,
    'val_ratio'     : 0.15,
    'model_name'    : 'efficientnet_b0',
    'num_classes'   : 2,
    'pretrained'    : True,
    'image_size'    : 224,
    'epochs'        : 15,
    'batch_size'    : 32,
    'learning_rate' : 1e-4,
    'weight_decay'  : 1e-4,
    'patience'      : 5,
    'class_names'   : ['colon_aca', 'colon_n'],
    'device'        : 'cuda' if torch.cuda.is_available() else 'cpu',
    'save_dir'      : SAVE_DIR,
    'model_filename': 'efficientnet_b0_colon.pth',
}

for k, v in CONFIG.items():
    print(f'  {k:20s}: {v}')

## Cell 5 — Load Dataset from HuggingFace

In [ ]:
# Upload dataset zip from local machine to Colab
from google.colab import files
import zipfile, os

# Step 1: First zip your colon_image_sets folder locally
# Run this in PowerShell on your machine:
# Compress-Archive -Path "data\raw\colon_image_sets" -DestinationPath "colon_image_sets.zip"

# Step 2: Upload the zip here
print("Select colon_image_sets.zip to upload...")
uploaded = files.upload()   # click and select the zip file

# Step 3: Extract
with zipfile.ZipFile('colon_image_sets.zip', 'r') as z:
    z.extractall('/content/data/')

print("Extracted!")
print(os.listdir('/content/data/colon_image_sets/'))

In [ ]:
# Transforms — defined here so available for dataset loading
import torch
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(90),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

print('Transforms defined.')

## Cell 6 — Train/Val/Test Split

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split
from torchvision import transforms

DATA_DIR = '/content/data/colon_image_sets'

# Verify structure
import os
for cls in os.listdir(DATA_DIR):
    n = len(os.listdir(f'{DATA_DIR}/{cls}'))
    print(f'{cls}: {n} images')

# Full dataset using ImageFolder — automatically uses folder names as labels
full_dataset = ImageFolder(root=DATA_DIR, transform=val_transform)
print(f'\nClasses: {full_dataset.classes}')
print(f'Total  : {len(full_dataset)} images')

# Split 70/15/15
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val   = int(0.15 * n_total)
n_test  = n_total - n_train - n_val

train_dataset, val_dataset, test_dataset = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

# Apply train transforms to train split
train_dataset.dataset.transform = train_transform

print(f'Train : {len(train_dataset)}')
print(f'Val   : {len(val_dataset)}')
print(f'Test  : {len(test_dataset)}')

# Update CONFIG class names to match folder names
CONFIG['class_names'] = full_dataset.classes
print(f'Class names: {CONFIG["class_names"]}')

## Cell 7 — Dataset & DataLoaders

In [ ]:
from torch.utils.data import DataLoader, Subset
import torch

# train_dataset, val_dataset, test_dataset already created in Cell 6
# Just create DataLoaders here

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

# Verify
images, labels = next(iter(train_loader))
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'Batch shape   : {images.shape}')
print(f'Labels sample : {labels[:8].tolist()}')
print(f'Class names   : {CONFIG["class_names"]}')

## Cell 8 — Build EfficientNet-B0 Model

In [ ]:
import timm
import torch.nn as nn

class ColonClassifier(nn.Module):
    def __init__(self, num_classes=2, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained, num_classes=0)
        n_features = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(256, num_classes),
        )
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False
        print('Backbone frozen')
    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        print('Backbone unfrozen')
    def forward(self, x):
        return self.classifier(self.backbone(x))

device = CONFIG['device']
model  = ColonClassifier(num_classes=2, pretrained=True).to(device)
total  = sum(p.numel() for p in model.parameters())
print(f'Parameters : {total:,}')
print(f'Device     : {device}')

## Cell 9 — Training Setup

In [ ]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
)
print('Optimizer, scheduler and loss function ready.')

## Cell 10 — Training & Validation Functions

In [ ]:
from tqdm.notebook import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(loader, desc='Training', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct      += (outputs.argmax(1) == labels).sum().item()
        total        += labels.size(0)
    return running_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in tqdm(loader, desc='Evaluating', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        preds    = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss/total, correct/total, all_preds, all_labels

print('Functions defined.')

## Cell 11 — Training Loop

In [ ]:
import time

history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}
best_val_acc    = 0.0
patience_count  = 0
best_model_path = f"{CONFIG['save_dir']}/{CONFIG['model_filename']}"

print(f"Training for {CONFIG['epochs']} epochs on {device}")
print('='*60)

for epoch in range(1, CONFIG['epochs']+1):
    t0 = time.time()

    if epoch == 4:
        model.unfreeze_backbone()
        for pg in optimizer.param_groups:
            pg['lr'] = CONFIG['learning_rate'] / 10

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc, _, _ = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch:2d}/{CONFIG["epochs"]}  '
          f'Train Loss:{train_loss:.4f} Acc:{train_acc:.4f}  '
          f'Val Loss:{val_loss:.4f} Acc:{val_acc:.4f}  '
          f'Time:{time.time()-t0:.1f}s')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'config': CONFIG,
        }, best_model_path)
        print(f'  --> Best model saved (val_acc={val_acc:.4f})')
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= CONFIG['patience']:
            print(f'Early stopping at epoch {epoch}')
            break

print('='*60)
print(f'Training complete. Best Val Acc: {best_val_acc:.4f}')

## Cell 12 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

x = range(1, len(history['train_loss'])+1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(x, history['train_loss'], 'b-o', label='Train')
ax1.plot(x, history['val_loss'],   'r-o', label='Val')
ax1.set_title('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(x, [a*100 for a in history['train_acc']], 'b-o', label='Train')
ax2.plot(x, [a*100 for a in history['val_acc']],   'r-o', label='Val')
ax2.set_title('Accuracy (%)'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B0 Training', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/training_curves.png", dpi=120, bbox_inches='tight')
plt.show()

## Cell 13 — Test Set Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns

checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded best model from epoch {checkpoint['epoch']}")

test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
f1 = f1_score(test_labels, test_preds, average='weighted')
cm = confusion_matrix(test_labels, test_preds)

print(f'Test Accuracy : {test_acc*100:.2f}%')
print(f'F1 Score      : {f1:.4f}')
print(classification_report(test_labels, test_preds, target_names=CONFIG['class_names']))

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CONFIG['class_names'],
            yticklabels=CONFIG['class_names'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — Test Acc: {test_acc*100:.2f}%  F1: {f1:.4f}',
             fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/confusion_matrix.png", dpi=120, bbox_inches='tight')
plt.show()

## Cell 14 — Grad-CAM Preview

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np

target_layers = [model.backbone.conv_head]
cam = GradCAM(model=model, target_layers=target_layers)

images_batch, labels_batch = next(iter(test_loader))
images_batch = images_batch[:8].to(device)

def denormalize(t):
    mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
    std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
    return (t.cpu()*std+mean).clamp(0,1)

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i in range(8):
    img_t  = images_batch[i:i+1]
    label  = labels_batch[i].item()
    grayscale = cam(input_tensor=img_t, targets=[ClassifierOutputTarget(label)])
    img_np = denormalize(images_batch[i]).permute(1,2,0).numpy().astype(np.float32)
    cam_img = show_cam_on_image(img_np, grayscale[0], use_rgb=True)
    axes[0,i].imshow(img_np)
    axes[0,i].set_title(CONFIG['class_names'][label], fontsize=7)
    axes[0,i].axis('off')
    axes[1,i].imshow(cam_img)
    axes[1,i].set_title('Grad-CAM', fontsize=7)
    axes[1,i].axis('off')

plt.suptitle('Grad-CAM Heatmaps', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f"{CONFIG['save_dir']}/gradcam_preview.png", dpi=120, bbox_inches='tight')
plt.show()

## Cell 15 — Save Summary

In [ ]:
import json

with open(f"{CONFIG['save_dir']}/training_history.json", 'w') as f:
    json.dump(history, f, indent=2)

print('='*60)
print('TRAINING COMPLETE')
print('='*60)
print(f'Best Val Acc : {best_val_acc*100:.2f}%')
print(f'Test Acc     : {test_acc*100:.2f}%')
print(f'Test F1      : {f1:.4f}')
print(f'Saved to     : {SAVE_DIR}')
print('Files: efficientnet_b0_colon.pth, training_curves.png,')
print('       confusion_matrix.png, gradcam_preview.png')
print('='*60)
print('Next step: Download .pth file and use in Stage 5 Dashboard')